<!-- nav -->
[← 8 · TypeScript Types](08_04_TypeScript_Types.ipynb) · [Home](../../README.md) · [8 · TypeScript RegExp →](08_06_TypeScript_RegExp.ipynb)

# 8 · TypeScript Patterns

ECMAScript takes values apart where it binds or assigns names, by destructuring:

```text
const swap = ([a, b]) => [b, a];
function label({ id, name: title = "untitled", ...rest }) { ... }
[a, b] = [b, a];
```

`[a, b]` and `{ id, name: title = "untitled", ...rest }` are patterns: they look like an array and an object, but take
one apart, binding `a`, `b`, `id`, `title` and `rest`. ECMA-262 calls them binding and assignment patterns (14.3.3,
"Destructuring Binding Patterns"; 13.15.5, "Destructuring Assignment"). `TypeScript Patterns` is their dialect, its kinds
typescript-estree's, `arraypattern`, `objectpattern`, `assignmentpattern` (a default), `restelement` and TypeScript's
`tsparameterproperty`, named and shaped as mbse-programs has them
([CAP-277: TypeScript's types and destructuring patterns as data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-277.md)).
Its `Term` lists, beside them, the TypeScript5 terms a target may be: a name, a member, an object's property, a hole.

This page builds patterns from a single name up, in the builders' fluent calls, prints each as TypeScript writes it,
finds where TypeScript5's terms hold one, shows what each binds, validates them, and reads the editions that have its
kinds. Like the rest of Part I, it starts from `dialects()`; `pattern` holds a pattern's spec in the dialect's holder,
`S.TypeScript.Patterns.Expression`, and `name` writes TypeScript5's `identifier` (tutorial 4). No reader here reads
TypeScript's text, in either language: reading TypeScript is mbse-programs' (tutorial 3).

In [1]:
import assert from "node:assert/strict";
import { Domains, Standards, Symbolics, Terms } from "@mbse/expressions/Framework";
import * as TypeScript5 from "@mbse/expressions/TypeScript/TypeScript5";
import * as TypeScriptPatterns from "@mbse/expressions/TypeScript/Patterns";
import { dialects } from "./toolkit.js";

const store = dialects();
const S = store.Expressions;
const PATTERNS = TypeScriptPatterns.Expressions.DIALECT;
const [ToText, binds] = [TypeScriptPatterns.Text.ToText, TypeScriptPatterns.Expressions.boundNames];
const name = (text: string) => (e: any) => e.identifier((i: any) => i.name(text)); // TypeScript5's identifier, `text`

/** The pattern `spec` writes, held by its dialect's holder, written in a transaction of its own. */
function pattern(spec: any): any {
  return store.transaction(() => S.TypeScript.Patterns.Expression((e: any) => e.root(spec)));
}

/** A pattern's text, as TypeScript writes it, its kind, and the names it binds. */
function show(held: any): void {
  const term = Terms.term(held);
  console.log(ToText(held).padEnd(46), Terms.kindOf(term)[0].padEnd(17), "binds:", [...binds(term)].sort().join(", ") || "none");
}

<a id="T08.1"></a>
## 08.1 · A name is the smallest pattern

The smallest target is a name: `a`, TypeScript5's `identifier`, which `TypeScript Patterns`' `Term` lists, so its holder
holds one. Where it is a parameter, it may carry an annotation, a type of `TypeScript Types` (the previous page), and
be optional; a pattern binds its names, which the dialect's `boundNames` gives:

In [2]:
for (const spec of [name("a"),
  (e: any) => e.identifier((i: any) => i.name("a").optional(true).typeAnnotation(
    (t: any) => t.typeAnnotation((k: any) => k.tsnumberkeyword((h: any) => h))))]) {
  const held = pattern(spec);
  show(held);
  console.log(" ".repeat(46), "of", Terms.dialectOf(Terms.term(held)), "| owner:", PATTERNS.owner(Terms.term(held)).name());
}

a                                              identifier        binds: a


                                               of Expressions.TypeScript.Patterns.Term | owner: TypeScript5


a?: number                                     identifier        binds: a


                                               of Expressions.TypeScript.Patterns.Term | owner: TypeScript5


<a id="T08.2"></a>
## 08.2 · Arrays, objects, defaults and rests

An **array** pattern, `[a, b]`, binds a sequence's items in order, its `elements`; a hole, `[a, , c]`, is an `elision`,
which skips one. An **object** pattern, `{ id, name: title }`, binds properties by their keys, its `properties` each
TypeScript5's `property`, a key and the target it binds, shorthand where they are the same name. A **default**,
`title = "untitled"`, is an `assignmentpattern`, a target `left` and an expression `right`, used where the value is
`undefined`; a **rest**, `...rest`, a `restelement`, binds what is left. They nest; the printer writes an object
pattern at the root in parentheses, as a statement holds one:

In [3]:
const title = (d: any) => d.assignmentpattern((a: any) => a.left(name("title")).right(
  (r: any) => r.literal((l: any) => l.value((v: any) => v.as_scalar("untitled")).quote('"'))));
const rest = (d: any) => d.restelement((r: any) => r.argument(name("rest")));
const specs = [
  (p: any) => p.arraypattern((a: any) => a.elements([name("a"), name("b")])),
  (p: any) => p.arraypattern((a: any) => a.elements([name("a"), (e: any) => e.elision((h: any) => h), name("c")])),
  (p: any) => p.objectpattern((o: any) => o.properties([
    (q: any) => q.property((s: any) => s.propertyKind("init").shorthand(true).key(name("id")).value(name("id"))),
    (q: any) => q.property((s: any) => s.propertyKind("init").key(name("name")).value(name("title")))])),
  title,
  (p: any) => p.objectpattern((o: any) => o.properties([
    (q: any) => q.property((s: any) => s.propertyKind("init").shorthand(true).key(name("id")).value(name("id"))),
    (q: any) => q.property((s: any) => s.propertyKind("init").key(name("name")).value(title)),
    rest])),
  (p: any) => p.arraypattern((a: any) => a.elements([
    (e: any) => e.arraypattern((b: any) => b.elements([name("x"), name("y")])), rest])),
];
for (const spec of specs) show(pattern(spec));

[a, b]                                         arraypattern      binds: a, b


[a,, c]                                        arraypattern      binds: a, c


({ id, name: title })                          objectpattern     binds: id, title


title = "untitled"                             assignmentpattern binds: title


({ id, name: title = "untitled", ...rest })    objectpattern     binds: id, rest, title


[[x, y], ...rest]                              arraypattern      binds: rest, x, y


<a id="T08.3"></a>
## 08.3 · Where TypeScript5 holds one, and what it binds there

A pattern binds names through what holds it: none of its kinds is a binder of its own (tutorial 7). A function's and
an arrow's parameters are patterns, which the function binds within its body, so `([a, b]) => [b, a]` needs nothing
from its scope. An assignment's target, `[a, b] = [b, a]`, is a pattern too, but it binds nothing: it assigns names
its scope already has, references, so its names are free. TypeScript5's `Term` lists the array, object and default
patterns, which may stand where an expression does in an assignment's target; and TypeScript5 reads `TypeScript
Patterns`' terms, as each dialect of the family reads the others' (`reads()`). The properties that hold one are read
from the store:

In [4]:
const swap = (b: any) => b.arrayexpression((a: any) => a.elements([ // [b, a]: the framework's list, of the framework's terms
  (t: any) => t.typescript5(name("b")), (t: any) => t.typescript5(name("a"))]));
const pair = (p: any) => p.arraypattern((a: any) => a.elements([name("a"), name("b")])); // [a, b]
const [arrow, assign]: any[] = store.transaction(() => [
  S.TypeScript.TypeScript5.Expression((e: any) => e.root((u: any) => u.arrowfunctionexpression((f: any) => f
    .params([pair]).body(swap)))),
  S.TypeScript.TypeScript5.Expression((e: any) => e.root((u: any) => u.assignmentexpression((a: any) => a
    .left(pair).operator("=").right(swap)))),
]);
for (const host of [arrow, assign]) {
  console.log(TypeScript5.Text.ToText(host).padEnd(20), Terms.kindOf(Terms.term(host))[0].padEnd(23), "free:",
    [...Symbolics.free(host, TypeScript5.Expressions.DIALECT)].sort().join(", ") || "none");
}
const patterns = store.registered(PATTERNS.union);
const own = TypeScriptPatterns.Expressions.TAGS as readonly string[];
const host = store.registered(TypeScript5.Expressions.DIALECT.union).branches as any[];
console.log("held by:", host.filter((b) => !own.includes(b.name)).flatMap((b) => [...b.schema.properties.entries()]
  .filter(([, p]: [string, any]) => p.schema === patterns || p.schema.element === patterns)
  .map(([n]: [string, any]) => `${b.name}.${n}`)).join(", "));
console.log("listed by TypeScript5:", host.filter((b) => own.includes(b.name)).map((b) => b.name).join(", "));
console.log("TypeScript5 reads", TypeScript5.Expressions.DIALECT.reads().map((d: any) => d.name()).join(", "));

([a, b]) => [b, a]   arrowfunctionexpression free: none


[a, b] = [b, a]      assignmentexpression    free: a, b


held by: functionexpression.params, arrowfunctionexpression.params, tsemptybodyfunctionexpression.params, assignmentexpression.left


listed by TypeScript5: arraypattern, objectpattern, assignmentpattern


TypeScript5 reads TypeScript Types, TypeScript Patterns, TypeScript RegExp


A pattern alone, in its holder, is held by nothing that binds: `boundNames` says which of its names a function would
bind, but alone every name it holds is free, those as much as a default's and a member's object, which no holder
binds.

In [5]:
const every = pattern((p: any) => p.objectpattern((o: any) => o.properties([ // { k: [a, , ...r] = c, m: x.y, ...z }
  (q: any) => q.property((s: any) => s.propertyKind("init").key(name("k")).value((v: any) => v.assignmentpattern((a: any) => a
    .left((t: any) => t.arraypattern((r: any) => r.elements([name("a"), (e: any) => e.elision((h: any) => h),
      (e: any) => e.restelement((s: any) => s.argument(name("r")))])))
    .right(name("c"))))),
  (q: any) => q.property((s: any) => s.propertyKind("init").key(name("m")).value((v: any) => v.memberexpression(
    (m: any) => m.object(name("x")).property(name("y"))))),
  (q: any) => q.restelement((r: any) => r.argument(name("z")))])));
show(every);
console.log(" ".repeat(46), "free:", [...Symbolics.free(every, PATTERNS)].sort().join(", "));

({ k: [a,, ...r] = c, m: x.y, ...z })          objectpattern     binds: a, r, z


                                               free: a, c, r, x, z


<a id="T08.4"></a>
## 08.4 · Validated

Validation checks each kind as typescript-estree has it, in its own words: a default needs its target and its value,
a rest its target, a name must be one TypeScript can spell, an annotation's type is checked as `TypeScript Types`
checks it, and invalid syntax where a target stands, an `error` term, is reported. A refused pattern is kept, and
printed as written
([PRI-002: preserve, don't lose](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-002)):

In [6]:
const refused = [
  (p: any) => p.arraypattern((a: any) => a.elements([name("a b")])),
  (p: any) => p.arraypattern((a: any) => a.elements([name("a"), (e: any) => e.error((r: any) => r.text("1 +"))])),
  (p: any) => p.objectpattern((o: any) => o.typeAnnotation((a: any) => a.typeAnnotation((t: any) => t.tsuniontype(
    (u: any) => u.types([(k: any) => k.tsanykeyword((h: any) => h)]))))),
];
for (const spec of refused) {
  const held = pattern(spec);
  console.log(ToText(held).padEnd(13), PATTERNS.validate(held, ["a"]).join("; "));
}
assert.deepEqual(PATTERNS.validate(pattern(refused[0]), ["a"]), ["elements 0: an identifier's name cannot be 'a b'"]);

[a b]         elements 0: an identifier's name cannot be 'a b'


[a, 1 +]      elements 1: invalid syntax '1 +'


({}: any)     typeAnnotation typeAnnotation: a tsuniontype needs two types


<a id="T08.5"></a>
## 08.5 · Inferred as any, and not evaluated

A pattern is no value: a function call or an assignment matches it. Inference gives it `any`, and TypeScript5's
evaluator refuses each of `TypeScript Patterns`' kinds by the dialect's name; a name the pattern holds is TypeScript5's
kind, refused by TypeScript5's:

In [7]:
for (const held of [pattern(pair), every, pattern(name("a"))]) {
  try {
    TypeScript5.Evaluators.OfAny(held);
    throw new Error("evaluated");
  } catch (error) {
    assert.equal((error as Error).name, "NotImplementedError");
    const scope = Object.fromEntries([...Symbolics.free(held, PATTERNS)].map((n) => [n, Domains.Anything]));
    console.log(ToText(held).padEnd(46), "inferred:", PATTERNS.infer(held, scope).name(), "|", (error as Error).message);
  }
}

[a, b]                                         inferred: any | TypeScript Patterns arraypattern is not evaluated


({ k: [a,, ...r] = c, m: x.y, ...z })          inferred: any | TypeScript Patterns objectpattern is not evaluated


a                                              inferred: any | TypeScript5 identifier is not evaluated


<a id="T08.6"></a>
## 08.6 · Its editions as data

`TypeScript Patterns` covers TypeScript 5.0 to 5.9, each found by its published name, and each with all five of its
own kinds ([CAP-268: standards as tagged data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-268.md));
the TypeScript5 terms its `Term` lists are recorded there:

In [8]:
const dialect = store.Instances.Expressions.TypeScript.Patterns;
const editions = [...Standards.standards(dialect).keys()];
console.log(editions.length, "editions:", editions[0], "to", editions.at(-1));
console.log("TypeScript 5.0:", Standards.kinds(Standards.find(dialect, "TypeScript 5.0")!).map((kind: any) => kind.name.split(".").at(-1)).join(", "));

10 editions: TypeScript 5.0 to TypeScript 5.9


TypeScript 5.0: ArrayPattern, ObjectPattern, AssignmentPattern, RestElement, TSParameterProperty


## What you learned

- **A destructuring target is a term of its own dialect**, `TypeScript Patterns`: array, object, default, rest and
  parameter-property patterns, beside TypeScript5's names, members and properties that a target may be.
- **It binds through what holds it**: a function binds its parameters' names within it; an assignment's target assigns
  names its scope has, so they stay free.
- **TypeScript5 holds a pattern** in a function's parameters and an assignment's target, and lists the array, object
  and default patterns in its own `Term`.
- **Validation speaks typescript-estree's rules**; inference gives `any`, and evaluation refuses each kind by name.
- **Its editions are data**: TypeScript 5.0 to 5.9.

Next, `TypeScript RegExp`: ECMAScript's regular expressions.

<!-- nav -->
[← 8 · TypeScript Types](08_04_TypeScript_Types.ipynb) · [Home](../../README.md) · [8 · TypeScript RegExp →](08_06_TypeScript_RegExp.ipynb)